In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/Sentiment_classification

/content/drive/MyDrive/Sentiment_classification


Load processed IMDB Movie Review data

In [5]:
import importlib
from src import preprocess
importlib.reload(preprocess)
from src.preprocess import load_and_preprocess

texts , labels = load_and_preprocess(
    df_path = "/content/drive/MyDrive/Sentiment_classification/data/IMDB Dataset.csv",
    text_column = "review",
    label_column = "sentiment"
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
labels

array([1, 1, 1, ..., 0, 0, 0])

In [7]:
texts.head()

,review
0,one review mention watch 1 oz episod youll hoo...
1,wonder littl product film techniqu unassum old...
2,thought wonder way spend time hot summer weeke...
3,basic there famili littl boy jake think there ...
4,petter mattei love time money visual stun film...


Convert word to vector using TFIDF

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(texts, labels ,test_size = 0.2, random_state = 42)

In [129]:
from src import vectorizers
importlib.reload(vectorizers)

from src.vectorizers import get_tfidf_vectorizer

vectorizer = get_tfidf_vectorizer(max_features = 10000,ngram_range = (1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)



In [130]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3877760 stored elements and shape (40000, 10000)>

In [131]:
X_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 970815 stored elements and shape (10000, 10000)>

TFIDF with linear models

In [132]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

models = {
    "Logistic Regression" :LogisticRegression(),
    "Linear Support Vector Classifier": LinearSVC(),
    "Naive Bayes": GaussianNB(),
    "Light GBM": LGBMClassifier()
}

Train and Predict

In [133]:
from sklearn.metrics import accuracy_score
import time
import pandas as pd

results = []

for name, model in models.items():
  print(f"\n Training {name}")
  start = time.time()
  #naive bayes requires dense matrix
  if name == "Naive Bayes":
    model.fit(X_train_tfidf.toarray(), y_train)
    y_pred = model.predict(X_test_tfidf.toarray())
  else:
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

  end = time.time()
  acc = accuracy_score(y_pred, y_test)

  results.append({
      "Model":name,
      "Max_features":vectorizer.max_features,
      "ngram_range":vectorizer.ngram_range,
      "Accuracy":acc *100,
      "Time":end - start
  })

df_results = pd.DataFrame(results)
df_results



 Training Logistic Regression

 Training Linear Support Vector Classifier

 Training Naive Bayes

 Training Light GBM
[LightGBM] [Info] Number of positive: 19961, number of negative: 20039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 7.971392 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 759986
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 9989
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499025 -> initscore=-0.003900
[LightGBM] [Info] Start training from score -0.003900


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Max_features,ngram_range,Accuracy,Time
0,Logistic Regression,10000,"(1, 2)",89.19,1.353774
1,Linear Support Vector Classifier,10000,"(1, 2)",88.94,2.061418
2,Naive Bayes,10000,"(1, 2)",83.06,21.445782
3,Light GBM,10000,"(1, 2)",86.05,80.057923


Add results to csv

In [134]:
df_results.to_csv("results/tfidf_results.csv",mode = "a", header = False ,index = False)

RNN on tfidf data

In [122]:
import torch
#convert train data to tensor
X_train_tfidf_tensor = torch.tensor(X_train_tfidf.toarray(), dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32)

#convert test data to tensor
X_test_tfidf_tensor = torch.tensor(X_test_tfidf.toarray(), dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.float32)

In [123]:
from torch.utils.data import TensorDataset, DataLoader

#tensors to tensordatasets
X_trainset = TensorDataset(X_train_tfidf_tensor, y_train_tensor)
X_testset = TensorDataset(X_test_tfidf_tensor, y_test_tensor)

#Dataloaders
train_loader = DataLoader(X_trainset, batch_size = 64, shuffle = True)
test_loader = DataLoader(X_testset, batch_size = 64)

In [124]:
#RNN layer
import torch.nn as nn
import torch.optim as optim
class SentimentRNN(nn.Module):
  def __init__(self, input_size, hidden_size = 128,num_layers = 1):
    super().__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers,
                      batch_first = True)
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):

    outputs, _ = self.rnn(x)

    out = self.fc(outputs[:,-1,:])
    return out



In [125]:
sentiment_rnn = SentimentRNN(input_size = X_train_tfidf.shape[1])

criterion = nn.BCELoss()
optimizer = optim.Adam(sentiment_rnn.parameters())

In [126]:
#Train
epochs = 15
rnn_start = time.time()
for epoch in range(epochs):
  sentiment_rnn.train()
  running_loss = 0.0
  for xb, yb in train_loader:
    #set gradients to zero
    optimizer.zero_grad()
    #xb shape -> [batch_size, input_size]
    #model expects -> [batch_size, sequence_length, input_size]
    #add singlton value
    xb = xb.unsqueeze(1)
    #predict
    output = sentiment_rnn.forward(xb)
    #output shape -> [batch_size, 1] and bceloss expects values between 0 and 1
    #squeeze output ->[batch_size] and apply sigmoid to convert values to 0 and 1
    output = torch.sigmoid(output.squeeze())

    #calculate loss
    loss = criterion(output, yb)

    #calculate gradients through backward propagation
    loss.backward()

    #update gradients
    optimizer.step()

    running_loss+= loss.item()

  #Average train loss per epoch
  print(f"{epoch +1}/{epochs} and loss {running_loss/len(train_loader)}")

rnn_end = time.time()

1/15 and loss 0.3415646819472313
2/15 and loss 0.2333819954752922
3/15 and loss 0.2174481522679329
4/15 and loss 0.21041460874080659
5/15 and loss 0.20582498490810394
6/15 and loss 0.202857702845335
7/15 and loss 0.19997490756511688
8/15 and loss 0.19827942249178887
9/15 and loss 0.19704552144408227
10/15 and loss 0.19564357886910438
11/15 and loss 0.19487175497412682
12/15 and loss 0.1937827664554119
13/15 and loss 0.19278545904159547
14/15 and loss 0.19198992980122567
15/15 and loss 0.191241265976429


In [127]:
sentiment_rnn.eval()
with torch.no_grad():
  total_values = 0.0
  correct_values = 0.0

  for xb, yb in test_loader:
    xb = xb.unsqueeze(1)
    output = sentiment_rnn.forward(xb)
    #split probabilities based on threshold 0.5
    predicted = (torch.sigmoid(output.squeeze())>0.5).float()
    correct_values +=(predicted == yb).sum().item()
    total_values += yb.size(0)
  acc = correct_values/total_values*100
  print(f"Accuracy: {acc}")



Accuracy: 87.13


In [128]:
rnn_result = pd.DataFrame([{
    "Model": "RNN(relu)",
    "Max_features":vectorizer.max_features,
    "ngram_range":vectorizer.ngram_range,
    "Accuracy": acc,
    "Time":rnn_end - rnn_start
}])